# Inspect the Symptoms from BioPortal

In [1]:
import pandas as pd 
import re
# Set pandas display options to show all content
pd.set_option('display.max_rows', None)  # Show all rows
pd.set_option('display.max_columns', None)  # Show all columns
pd.set_option('display.max_colwidth', None)  # Show full content in each cell (no truncation)
pd.set_option('display.width', None)  # Allow unlimited width for display
pd.set_option('display.expand_frame_repr', False)  # Don't wrap to multiple pages


In [2]:
data = pd.read_csv("bio_portal_symptoms/symptom_tree.csv")
# copy
data_copy = data.copy()
print("Columns:", data.columns.tolist())
print("Number of rows:", len(data))
print("Unique Symptoms 'Pref Label':", data['prefLabel'].nunique())


Columns: ['path', 'level', 'prefLabel', 'synonym', 'definition']
Number of rows: 895
Unique Symptoms 'Pref Label': 893


# Quick Overview with Duplicates Inspection

In [3]:
data.head(3)

,path,level,prefLabel,synonym,definition
0,symptom,0,symptom,NaN,"A symptom is a perceived change in function, sensation, loss, disturbance or appearance reported by a patient indicative of a disease."
1,symptom/musculoskeletal system symptom,1,musculoskeletal system symptom,NaN,NaN
2,symptom/musculoskeletal system symptom/torticollis,2,torticollis,twisted neck|wry neck,NaN


## Duplicate Symtoms 

There are duplicates because of there being the same symptom appearing in different subclasses.

In [4]:
# Find duplicate prefLabel values
duplicate_labels = data[data.duplicated(subset=['prefLabel'], keep=False)]
print(f"Number of rows with duplicate prefLabels: {len(duplicate_labels)}")
print(f"Expected: 895 rows - 893 unique = 2 duplicates")
print(f"\nDuplicate prefLabel values and their counts:")
print(duplicate_labels['prefLabel'].value_counts())
print(f"\nDetailed view of all rows with duplicate prefLabels:")
duplicate_labels

Number of rows with duplicate prefLabels: 4
Expected: 895 rows - 893 unique = 2 duplicates

Duplicate prefLabel values and their counts:
prefLabel
muscle necrosis       2
hepatosplenomegaly    2
Name: count, dtype: int64

Detailed view of all rows with duplicate prefLabels:


,path,level,prefLabel,synonym,definition
28,symptom/musculoskeletal system symptom/muscle symptom/muscle necrosis,3,muscle necrosis,NaN,NaN
43,symptom/musculoskeletal system symptom/soft tissue necrosis/muscle necrosis,3,muscle necrosis,NaN,NaN
75,symptom/hemic and immune system symptom/immune system symptom/spleen symptom/hepatosplenomegaly,4,hepatosplenomegaly,NaN,NaN
120,symptom/digestive system symptom/liver symptom/hepatosplenomegaly,3,hepatosplenomegaly,NaN,NaN


In [5]:
# Remove duplicated prefLabel rows, keeping the first occurrence
data = data.drop_duplicates(subset=['prefLabel'], keep='first')

# Print new shape after removing duplicates
print(f"Shape after removing duplicate prefLabels: {data.shape}")

# Print the remaining (now unique) prefLabels that were previously duplicated
print("Previously duplicated prefLabels now present only once:")
dupe_labels_set = set(duplicate_labels['prefLabel'])
remaining_dupes = data[data['prefLabel'].isin(dupe_labels_set)]
print(remaining_dupes[['prefLabel', 'path', 'level']])


Shape after removing duplicate prefLabels: (893, 5)
Previously duplicated prefLabels now present only once:
             prefLabel                                                                                             path  level
28     muscle necrosis                            symptom/musculoskeletal system symptom/muscle symptom/muscle necrosis      3
75  hepatosplenomegaly  symptom/hemic and immune system symptom/immune system symptom/spleen symptom/hepatosplenomegaly      4


## Look at Rows with Synonyms

In [6]:
data[data['synonym'].notna()].head(10)

,path,level,prefLabel,synonym,definition
2,symptom/musculoskeletal system symptom/torticollis,2,torticollis,twisted neck|wry neck,NaN
17,symptom/musculoskeletal system symptom/muscle symptom/muscle weakness/limb weakness,4,limb weakness,extremities weakness,NaN
61,symptom/hemic and immune system symptom/immune system symptom/anaphylactic shock,3,anaphylactic shock,anaphylaxis,An acute allergic reaction to an antigen to which the body has become hypersensitive.
63,symptom/hemic and immune system symptom/immune system symptom/lymphatic system symptom/enlargement of lymph nodes,4,enlargement of lymph nodes,swelling of lymph nodes|swollen lymph glands,NaN
64,symptom/hemic and immune system symptom/immune system symptom/lymphatic system symptom/enlargement of lymph nodes/lymphadenopathy,5,lymphadenopathy,adenopathy,NaN
71,symptom/hemic and immune system symptom/immune system symptom/lymphatic system symptom/lymphadenitis/bubo,5,bubo,buboes,NaN
76,symptom/hemic and immune system symptom/immune system symptom/spleen symptom/splenomegaly,4,splenomegaly,esplenomegaly,NaN
90,symptom/hemic and immune system symptom/hemic system symptom/anemia,3,anemia,anaemia,"doid/symp duplicate - reviewed 10/2022, both DO & SYMP"
95,symptom/hemic and immune system symptom/hemic system symptom/hemolysis,3,hemolysis,haemolysis,NaN
96,symptom/hemic and immune system symptom/hemic system symptom/hyperemia,3,hyperemia,hyperaemia|hyperemic,Hyperemia is a hemic system symptom consisting of an excess of blood in a body part as from an increased flow of blood due to vasodilation.


In [7]:
# How are symptoms' synonyms separated?
data.iloc[95][['prefLabel','synonym']]#.split('|')

prefLabel               hyperemia
synonym      hyperaemia|hyperemic
Name: 96, dtype: object

## Find Prefered Labels with more than a word. 

In [8]:
data['prefLabel'] = data['prefLabel'].astype(str)


In [9]:
# Split data into two dataframes: 
    # composed_data (multi-word 'prefLabel') 
    # and simple_data (single word)

# Count words in 'prefLabel' for each row and add as a new column
data['word_count_in_prefLabel'] = data['prefLabel'].str.split().apply(len)

composed_data = data[data['word_count_in_prefLabel'] > 1]
simple_data = data[data['word_count_in_prefLabel'] == 1]

print("Number of composed symptom names: ", len(composed_data))
print("Number of simple symptom names: ", len(simple_data))
data.head()

Number of composed symptom names:  634
Number of simple symptom names:  259


,path,level,prefLabel,synonym,definition,word_count_in_prefLabel
0,symptom,0,symptom,NaN,"A symptom is a perceived change in function, sensation, loss, disturbance or appearance reported by a patient indicative of a disease.",1
1,symptom/musculoskeletal system symptom,1,musculoskeletal system symptom,NaN,NaN,3
2,symptom/musculoskeletal system symptom/torticollis,2,torticollis,twisted neck|wry neck,NaN,1
3,symptom/musculoskeletal system symptom/loss of height,2,loss of height,NaN,NaN,3
4,symptom/musculoskeletal system symptom/abnormal posture,2,abnormal posture,NaN,NaN,2


# Build Symptom Dictionary

In [10]:
# Normalize and clean 'prefLabel' column:
#   - Make lowercase
#   - Remove extra whitespace
#   - Remove punctuation
data['prefLabel'] = (
    data['prefLabel']
    .str.lower()
    .str.replace(r'[^\w\s]', '', regex=True)    # remove punctuation
    .str.replace(r'\s+', ' ', regex=True)
    .str.strip()
)

# Normalize and clean 'synonym' column for non-NaN rows, if present:
#   - Make lowercase
#   - Remove extra whitespace
#   - Remove punctuation
if 'synonym' in data.columns:
    # Define a normalizing function for individual synonyms
    def normalize_synonym(s):
        return (
            s.lower()
            .replace('\n', '')
            .replace('\t', '')
            .replace('\r', '')
            .strip()
        )
    def clean_and_split_synonyms(syn_str):
        if pd.isna(syn_str):
            return []
        # Split by '|' into list
        syns = [syn for syn in syn_str.split('|')]
        # Normalize each synonym and remove punctuation/extra whitespace
        normalized = [
            normalize_synonym(
                re.sub(r'[^\w\s]', '', syn)  # remove punctuation
                .replace('\u200b', '')       # remove zero-width spaces if present
            ).replace('  ', ' ')             # replace double spaces with single
            for syn in syns
        ]
        # Remove empty strings
        return [s for s in normalized if s]

    data['synonym'] = data['synonym'].apply(clean_and_split_synonyms)

# Add a unique id column for each symptom
# The zfill(4) method pads each number with leading zeros so that string has at least 4 digits ('1' -> '0001')
data['id'] = ['s' + str(i).zfill(4) for i in range(1, len(data) + 1)]


In [11]:
data.head()

,path,level,prefLabel,synonym,definition,word_count_in_prefLabel,id
0,symptom,0,symptom,[],"A symptom is a perceived change in function, sensation, loss, disturbance or appearance reported by a patient indicative of a disease.",1,s0001
1,symptom/musculoskeletal system symptom,1,musculoskeletal system symptom,[],NaN,3,s0002
2,symptom/musculoskeletal system symptom/torticollis,2,torticollis,"[twisted neck, wry neck]",NaN,1,s0003
3,symptom/musculoskeletal system symptom/loss of height,2,loss of height,[],NaN,3,s0004
4,symptom/musculoskeletal system symptom/abnormal posture,2,abnormal posture,[],NaN,2,s0005


In [14]:
data.columns

Index(['path', 'level', 'prefLabel', 'synonym', 'definition',
       'word_count_in_prefLabel', 'id'],
      dtype='object')

## Save The Base Sympton Dictionary

In [ ]:
# data.to_csv("base_symptom_dict.csv", index=False)